In [ ]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
valid_dataset = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)
valid_loader = DataLoader(valid_dataset, batch_size=32)

learning_rate = 0.1
n_epochs=20

model = nn.Sequential(
	nn.Linear(in_features=8, out_features=30), 
	nn.ReLU(),
	nn.Linear(in_features=30, out_features=50), 
	nn.ReLU(),
	nn.Linear(in_features=50, out_features=1)
)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=learning_rate)
metric = torchmetrics.MeanAbsoluteError()

history = {
	'loss' : [],
	'metric' : [],
}
for epoch in range(n_epochs):
	total_loss = 0
	metric.reset()
	for X_batch, y_batch in train_loader:
		y_pred = model(X_batch)
		loss = criterion(y_pred, y_batch)
		total_loss += loss.item()
		loss.backward()
		optimizer.step()
		optimizer.zero_grad()
		metric.update(y_pred, y_batch)
	
	avg_loss = total_loss / len(train_loader)
	history['loss'].append(avg_loss)

	avg_metric = metric.compute().item()
	history['metric'].append(avg_metric)

	print(f'Epoch: {epoch+1}/{n_epochs}, loss: {avg_loss}, metric: {avg_metric}')

	# if epoch>=3:
	# 	break

plt.plot(np.arange(n_epochs) + 1, history['metric'], linestyle='--', marker='.')

plt.grid()
plt.xlabel('Epochs')
plt.ylabel(f'{metric.__class__.__name__}')
plt.show()

In [ ]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
valid_dataset = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)
valid_loader = DataLoader(valid_dataset, batch_size=32)

learning_rate = 0.1
n_epochs=20
device = 'cpu' # 'cuda'

model = nn.Sequential(
	nn.Linear(in_features=8, out_features=30), 
	nn.Sigmoid(),
	nn.Linear(in_features=30, out_features=50), 
	nn.Sigmoid(),
	nn.Linear(in_features=50, out_features=1)
).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=learning_rate)
metric = torchmetrics.MeanAbsoluteError().to(device)

history = {
	'loss' : [],
	'train_metric' : [],
	'valid_metric' : [],
}
for epoch in range(n_epochs):
	# Training 
	total_loss = 0
	metric.reset()
	for X_batch, y_batch in train_loader:
		X_batch, y_batch = X_batch.to(device), y_batch.to(device)
		model.train()
		y_pred = model(X_batch)
		loss = criterion(y_pred, y_batch)
		total_loss += loss.item()
		loss.backward()
		optimizer.step()
		optimizer.zero_grad()
		metric.update(y_pred, y_batch)
	
	avg_loss = total_loss / len(train_loader)
	history['loss'].append(avg_loss)

	avg_metric_train = metric.compute().item()
	history['train_metric'].append(avg_metric_train)

	# Evaluation 
	model.eval()
	metric.reset()
	for X_batch, y_batch in valid_loader:
		X_batch, y_batch = X_batch.to(device), y_batch.to(device)
		with torch.no_grad():
			y_pred = model(X_batch)
			metric.update(y_pred, y_batch)

	avg_metric_valid = metric.compute().item()
	history['valid_metric'].append(avg_metric_valid)

	print(f'Epoch: {epoch+1}/{n_epochs}, loss: {avg_loss}, train_metric: {round(avg_metric_train,3)}, valid_metric: {round(avg_metric_valid,3)}, ')

	# if epoch>=2:
	# 	break

plt.plot(np.arange(n_epochs) + 1, history['train_metric'], linestyle='--', color='r', marker='.', label='Train')
plt.plot(np.arange(n_epochs) + 1, history['valid_metric'], linestyle='--', color='b', marker='.', label='Valid')
plt.legend()
plt.grid()
plt.xlabel('Epochs')
plt.ylabel(f'{metric.__class__.__name__}')
plt.show()